In [ ]:
import sys, subprocess, glob, os, shutil, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.6", "catboost>=1.2"])

def find_one(filename):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{filename} not found under /kaggle/input")
    return hits[0]

# source package — globs for run_bakeoff.py regardless of dataset version slug
SRC = find_one("run_bakeoff.py").split("/models/")[0]
sys.path.insert(0, SRC)
print("SRC:", SRC)

SAMPLE = find_one("sample_5M_seed42.parquet")
FULL   = find_one("features_2026-05-18.parquet")
print("SAMPLE:", SAMPLE, "
FULL:", FULL)

# Copy study DBs to a writable dir (SQLite cannot open read-only /kaggle/input)
WORK = "/kaggle/working/studies"
os.makedirs(WORK, exist_ok=True)
for name in ("lightgbm", "catboost", "xgboost"):
    hits = sorted(glob.glob(f"/kaggle/input/**/study_{name}*.db", recursive=True))
    if not hits:
        raise FileNotFoundError(f"missing study DB for {name}")
    shutil.copy(hits[0], f"{WORK}/study_{name}.db")
print("studies:", os.listdir(WORK))

In [ ]:
from models.forecasting.run_bakeoff import run_bakeoff

# CatBoost only — MultiQuantile has no GPU kernel, runs CPU.
# Fresh kernel -> full 30 GB available for Pool creation.
report = run_bakeoff(
    sample_path=SAMPLE,
    full_path=FULL,
    models=["catboost"],
    n_trials=0,
    out_dir="/kaggle/working/bakeoff",
    device="cpu", task_type="CPU",
    storage_dir=WORK,
)
print(json.dumps(report, indent=2))